# Ink & Type UI
This launcher uses the stronger original pretrained checkpoint. CPU works when a Colab GPU is unavailable.

In [ ]:
%pip -q install transformers==4.57.1 gradio==5.49.1 sentencepiece==0.2.2 opencv-python-headless
from pathlib import Path
ROOT=Path("/content/ocr2_results");ROOT.mkdir(exist_ok=True)

In [ ]:
from transformers import TrOCRProcessor,VisionEncoderDecoderModel
p=TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten",use_fast=False)
m=VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-handwritten",use_safetensors=False)
p.save_pretrained(ROOT/"base_model");m.save_pretrained(ROOT/"base_model");del m
(ROOT/"reports").mkdir(exist_ok=True)

In [ ]:
(ROOT/'app.py').write_text('"""Ink & Type: a local/Colab interface for the delivered OCR checkpoint."""\nimport csv, json, os, tempfile, time, threading\nfrom functools import lru_cache\nfrom pathlib import Path\nimport gradio as gr\nimport torch\nfrom PIL import Image, ImageDraw\nfrom transformers import TrOCRProcessor, VisionEncoderDecoderModel\nfrom ocr_page import prep, segment\n\nROOT=Path(__file__).resolve().parent\nMODEL=Path(os.environ.get(\'OCR_MODEL_PATH\',str(ROOT/(\'base_model\' if (ROOT/\'base_model/config.json\').exists() else \'model\'))))\nRUNS=Path(tempfile.gettempdir())/\'ink_type_outputs\'; RUNS.mkdir(exist_ok=True)\nDEVICE=\'cuda\' if torch.cuda.is_available() else (\'mps\' if torch.backends.mps.is_available() else \'cpu\')\nLOCK=threading.Lock()\n\n@lru_cache(maxsize=1)\ndef engine():\n    if not (MODEL/\'config.json\').exists():\n        raise RuntimeError(\'Model files are missing. Place the trained model folder beside app.py.\')\n    p=TrOCRProcessor.from_pretrained(MODEL,use_fast=False,local_files_only=True)\n    m=VisionEncoderDecoderModel.from_pretrained(MODEL,local_files_only=True).to(DEVICE).eval()\n    return p,m\n\ndef empty(): return \'\',None,[], \'Upload a page or choose a sample to begin.\',None\n\ndef recognize(image,mode,progress=gr.Progress()):\n    if image is None: return \'\',None,[],\'Choose an image first.\',None\n    if image.width*image.height>24_000_000: return \'\',None,[],\'This image is too large. Resize it below 24 megapixels and try again.\',None\n    try:\n        started=time.perf_counter(); progress(.03,desc=\'Preparing your page\')\n        if mode==\'Single text line\':\n            page=prep(image); boxes=[[0,0,page.width,page.height]]\n        else: page,boxes,_=segment(image)\n        if not boxes: return \'\',page,[],\'No text lines found. Try a brighter, closer image with dark writing.\',None\n        if len(boxes)>100: return \'\',page,[],\'Too many possible text lines. Crop to one clean, single-column page.\',None\n        annotated=page.copy(); draw=ImageDraw.Draw(annotated); crops=[]; text=[]\n        with LOCK,torch.inference_mode():\n            p,m=engine()\n            for i,box in enumerate(boxes):\n                progress((i+.1)/len(boxes),desc=f\'Reading line {i+1} of {len(boxes)}\')\n                crop=page.crop(box)\n                px=p(images=prep(crop),return_tensors=\'pt\').pixel_values.to(DEVICE)\n                ids=m.generate(px,max_new_tokens=128,num_beams=1)\n                line=p.batch_decode(ids,skip_special_tokens=True)[0]\n                text.append(line);crops.append((crop,f\'Line {i+1}: {line}\'))\n                draw.rectangle(box,outline=\'#245ae5\',width=max(2,page.width//700))\n                draw.text((box[0]+3,box[1]+2),str(i+1),fill=\'#163d9c\')\n        result=\'\\n\'.join(text); path=save_text(result)\n        return result,annotated,crops,f\'{len(boxes)} line(s) read in {time.perf_counter()-started:.1f}s. Review and correct the text before using it.\',path\n    except Exception as exc:\n        return \'\',None,[],f\'Could not read this image: {exc}\',None\n\ndef save_text(text):\n    if not text or not text.strip(): return None\n    folder=Path(tempfile.mkdtemp(prefix=\'transcription_\',dir=RUNS))\n    path=folder/\'transcription.txt\'; path.write_text(text,encoding=\'utf-8\'); return str(path)\n\nusing_base=MODEL.name==\'base_model\'\nmetric_path=ROOT/\'reports/corrected_metrics.json\' if using_base else ROOT/\'reports/metrics.json\'\nmetrics=json.loads(metric_path.read_text()) if metric_path.exists() else {}\nresult=metrics.get(\'baseline_test\' if using_base else \'finetuned_test\',{})\nmetric_rows=[[name,f\'{100*result[key]:.2f}%\' if key in result else \'Not measured\'] for name,key in [(\'Character error rate\',\'cer\'),(\'Word error rate\',\'wer\'),(\'Perfectly transcribed lines\',\'exact_line_accuracy\')]]\nexamples=[]\nmanifest=ROOT/\'data/manifest.csv\'\nif manifest.exists():\n    with manifest.open(encoding=\'utf-8\') as f:\n        for row in csv.DictReader(f):\n            if row[\'split\']==\'test\':\n                examples.append([str(ROOT/row[\'raw\']),\'Single text line\'])\n                if len(examples)==3:break\nassembled=ROOT/\'assembled_page_demo/input.png\'\nif assembled.exists(): examples.insert(0,[str(assembled),\'Full page\'])\n\nCSS=\'\'\'\nbody .gradio-container{box-sizing:border-box!important;width:100vw!important;min-width:0!important;}\nbody .gradio-container main{width:100%!important;max-width:100%!important;min-width:0!important;box-sizing:border-box!important;}\n\n.gradio-container {max-width:1200px!important; margin:auto!important; background:#f5f8fc!important; color:#152a43!important;}\n#masthead {border-bottom:1px solid #cbd5e2;padding:24px 2px 22px;margin-bottom:16px;}\n#masthead .wordmark{font-size:15px;font-weight:700;color:#245ae5;letter-spacing:.04em;}\n#masthead h1{font-family:Georgia,serif!important;font-size:42px!important;line-height:1.12;margin:12px 0;color:#152a43!important;}\n#masthead p{font-size:16px;color:#53657c;max-width:620px;line-height:1.6;}\n#notice{background:#fff7e7;border-left:3px solid #b9780a;border-radius:4px;padding:12px 16px;font-size:14px;color:#59410e;}\n#transcript textarea{font-family:Georgia,serif!important;font-size:19px!important;line-height:1.75!important;background:#fff!important;}\n#read-button{background:#245ae5!important;border:none!important;color:white!important;min-height:48px;font-size:16px;}\n#status{min-height:42px;color:#53657c;font-size:14px;}\n#footnote{border-top:1px solid #cbd5e2;padding-top:14px;margin-top:18px;color:#53657c;font-size:13px;}\nbutton:focus-visible,input:focus-visible,textarea:focus-visible{outline:3px solid #95b1ff!important;outline-offset:3px;}\n@media(max-width:700px){#masthead h1{font-size:30px!important;}#masthead{padding-top:14px;} .gradio-container{padding:12px!important;width:100%!important;max-width:100%!important;min-width:0!important;box-sizing:border-box!important;}}\n\'\'\'\ntheme=gr.themes.Base(primary_hue=\'blue\',neutral_hue=\'slate\').set(body_background_fill_dark=\'#f5f8fc\',body_text_color_dark=\'#152a43\',block_background_fill_dark=\'#ffffff\',block_label_text_color_dark=\'#53657c\',block_label_background_fill_dark=\'#f1f5fb\',block_title_text_color_dark=\'#152a43\',input_background_fill_dark=\'#ffffff\',table_text_color_dark=\'#152a43\',table_even_background_fill_dark=\'#ffffff\',table_odd_background_fill_dark=\'#f1f5fb\')\nwith gr.Blocks(title=\'Ink & Type — Handwriting reader\',css=CSS,theme=theme,analytics_enabled=False,delete_cache=(3600,86400)) as demo:\n    gr.HTML(\'<header id="masthead"><div class="wordmark">Ink & Type</div><h1>Your handwriting,<br>ready to edit.</h1><p>Bring a handwritten page into a text workspace. Read, compare, and keep the words that matter.</p></header>\')\n    gr.HTML(f\'<div id="notice"><strong>Review your transcription.</strong> Measured line-test character error rate: {100*result.get("cer",0):.2f}%. Full-page accuracy has not been measured.</div>\')\n    with gr.Tab(\'Read handwriting\'):\n        with gr.Row(equal_height=True):\n            with gr.Column(scale=1,min_width=300):\n                image=gr.Image(label=\'Your handwritten page\',type=\'pil\',sources=[\'upload\',\'clipboard\'],height=380)\n                mode=gr.Radio([\'Full page\',\'Single text line\'],value=\'Full page\',label=\'What are you uploading?\')\n                read=gr.Button(\'Read handwriting\',variant=\'primary\',elem_id=\'read-button\')\n                gr.Markdown(\'Use a clear **English, single-column** page. PNG, JPG or WebP. Keep the page upright and evenly lit.\')\n            with gr.Column(scale=1,min_width=300):\n                text=gr.Textbox(label=\'Editable transcription\',placeholder=\'Your text will appear here. You can correct it before downloading.\',lines=14,interactive=True,show_copy_button=True,elem_id=\'transcript\')\n                status=gr.Markdown(\'Upload a page or choose a sample to begin.\',elem_id=\'status\')\n                with gr.Row():\n                    export=gr.Button(\'Save edited text\')\n                    clear=gr.Button(\'Start again\')\n                download=gr.File(label=\'Download your text\',interactive=False)\n        if examples:\n            gr.Examples(examples=examples,inputs=[image,mode],label=\'Try a sample — test lines and one assembled page\',examples_per_page=4)\n        with gr.Accordion(\'Check what the model read\',open=False):\n            gr.Markdown(\'Blue boxes show the detected lines. Missing or merged boxes can cause missing text. Crop the image or switch to a single line if needed.\')\n            annotated=gr.Image(label=\'Detected lines\',interactive=False,height=350)\n            gallery=gr.Gallery(label=\'Line crops and readings\',columns=2,height=300,object_fit=\'contain\')\n    with gr.Tab(\'Model results\'):\n        gr.Markdown(\'### Measured results\\nActive model: **Original pretrained TrOCR-small**. Corrected fine-tuning regressed to 22.81% CER; the stronger unchanged checkpoint is used here.\')\n        gr.Dataframe(value=metric_rows,headers=[\'Metric\',\'Measured value\'],interactive=False)\n        gr.Markdown(f"**Test lines:** {result.get(\'samples\',\'Unavailable\')} · **Selected epoch:** {\'Not applicable\' if using_base else metrics.get(\'selected_epoch\',\'Unavailable\')}\\n\\nTraining used 6,482 lines, with 256 validation lines and 512 test lines. All selected raw images, preprocessed images and labels are included in the results package.\\n\\nThe starting checkpoint was already trained on IAM. This is not an independent unseen-corpus benchmark. Multi-column pages, tables and overlapping writing are unsupported.")\n        if metric_path.exists(): gr.File(value=str(metric_path),label=\'Download the measured metrics\',interactive=False)\n    gr.HTML(\'<footer id="footnote">English handwriting · Review before use · Your uploads are processed by the machine running this app.</footer>\')\n    outputs=[text,annotated,gallery,status,download]\n    read.click(lambda:(\' \',None,[],\'Reading your image…\',None),outputs=outputs,queue=False).then(recognize,inputs=[image,mode],outputs=outputs,concurrency_limit=1)\n    export.click(save_text,inputs=text,outputs=download)\n    clear.click(lambda:(None,\'Full page\',*empty()),outputs=[image,mode,*outputs],queue=False)\n    image.change(empty,outputs=outputs,queue=False)\n    mode.change(empty,outputs=outputs,queue=False)\n\ndef launch(port=7860):\n    # Mounting avoids creating a public Gradio sharing link in Colab.\n    from fastapi import FastAPI\n    import uvicorn\n    app=gr.mount_gradio_app(FastAPI(),demo.queue(),path=\'/\')\n    if \'google.colab\' in __import__(\'sys\').modules:\n        threading.Thread(target=uvicorn.run,args=(app,),kwargs={\'host\':\'0.0.0.0\',\'port\':port,\'log_level\':\'warning\'},daemon=True).start()\n        from google.colab import output\n        output.serve_kernel_port_as_iframe(port,height=1100)\n    else: uvicorn.run(app,host=\'127.0.0.1\',port=port)\n\nif __name__==\'__main__\':\n    engine()\n    launch(int(os.environ.get(\'PORT\',\'7860\')))\n',encoding='utf-8')

In [ ]:
(ROOT/'ocr_page.py').write_text("# Heuristic line separation + trained TrOCR. Clean single-column pages only.\nimport argparse,json\nfrom pathlib import Path\nimport cv2,numpy as np,torch\nfrom PIL import Image,ImageOps\nfrom transformers import TrOCRProcessor,VisionEncoderDecoderModel\ndef prep(im):\n    im=ImageOps.exif_transpose(im).convert('RGBA'); bg=Image.new('RGBA',im.size,'white'); bg.alpha_composite(im)\n    return ImageOps.autocontrast(bg.convert('L'),cutoff=.2).convert('RGB')\ndef segment(im):\n    im=prep(im); im.thumbnail((2400,3400)); gray=np.array(im.convert('L'))\n    ink=cv2.threshold(gray,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)[1]\n    horizontal=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((1,max(80,im.width//3)),np.uint8))\n    vertical=cv2.morphologyEx(ink,cv2.MORPH_OPEN,np.ones((max(80,im.height//3),1),np.uint8))\n    mask=cv2.subtract(cv2.subtract(ink,horizontal),vertical)\n    def rotate(a,angle,fill=0):\n        h,w=a.shape[:2]; return cv2.warpAffine(a,cv2.getRotationMatrix2D((w/2,h/2),angle,1),(w,h),borderValue=fill)\n    small=cv2.resize(mask,(max(1,im.width//3),max(1,im.height//3)))\n    angles=np.arange(-5,5.01,.5); scores=[np.square(rotate(small,a).sum(axis=1,dtype=np.float64)).sum() for a in angles]\n    angle=float(angles[int(np.argmax(scores))]) if mask.any() else 0.\n    mask=rotate(mask,angle); im=Image.fromarray(rotate(np.array(im),angle,(255,255,255)))\n    n,labels,stats,_=cv2.connectedComponentsWithStats(mask); clean=np.zeros_like(mask)\n    for i in range(1,n):\n        if stats[i,cv2.CC_STAT_AREA]>=6: clean[labels==i]=255\n    active=(np.count_nonzero(clean,axis=1)>=max(2,int(im.width*.001))).astype(np.uint8)\n    active=cv2.morphologyEx(active[:,None],cv2.MORPH_CLOSE,np.ones((7,1),np.uint8)).ravel()\n    edges=np.diff(np.pad(active.astype(int),(1,1))); boxes=[]\n    for y0,y1 in zip(np.flatnonzero(edges==1),np.flatnonzero(edges==-1)):\n        xs=np.flatnonzero(clean[y0:y1].any(axis=0))\n        if y1-y0>=5 and len(xs)>=8: boxes.append([max(0,int(xs[0])-8),max(0,int(y0)-5),min(im.width,int(xs[-1])+9),min(im.height,int(y1)+5)])\n    return im,boxes,angle\n@torch.inference_mode()\ndef transcribe(image,model_path,out):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    device='cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')\n    p=TrOCRProcessor.from_pretrained(model_path,use_fast=False); m=VisionEncoderDecoderModel.from_pretrained(model_path).to(device).eval()\n    im,boxes,angle=segment(Image.open(image));im.save(out/'deskewed.png');lines=[]\n    for i,box in enumerate(boxes):\n        crop=im.crop(box);crop.save(out/f'line_{i:03}.png')\n        px=p(images=prep(crop),return_tensors='pt').pixel_values.to(device)\n        text=p.batch_decode(m.generate(px,max_new_tokens=128,num_beams=1),skip_special_tokens=True)[0]\n        lines.append({'line':i,'box_on_deskewed_image':box,'text':text})\n    text='\\n'.join(r['text'] for r in lines);(out/'transcription.txt').write_text(text,encoding='utf-8')\n    (out/'layout.json').write_text(json.dumps({'rotation_degrees':angle,'lines':lines},indent=2))\n    return text\nif __name__=='__main__':\n    p=argparse.ArgumentParser();p.add_argument('image');p.add_argument('--model',default='model');p.add_argument('--out',default='page_output');a=p.parse_args()\n    print(transcribe(a.image,a.model,a.out))\n",encoding="utf-8")

In [ ]:
(ROOT/"reports/corrected_metrics.json").write_text('{\n  "status": "completed",\n  "baseline_test": {\n    "cer": 0.05920580853352456,\n    "wer": 0.1363040629095675,\n    "exact_line_accuracy": 0.408203125,\n    "samples": 512,\n    "loss": 10.868056878447533\n  },\n  "finetuned_test": {\n    "cer": 0.22808354248834709,\n    "wer": 0.3816076889471385,\n    "exact_line_accuracy": 0.02734375,\n    "samples": 512,\n    "loss": 1.9949353467673063\n  },\n  "selected_epoch": 1,\n  "selection_metric": "validation CER",\n  "history": [\n    {\n      "epoch": 1,\n      "train_loss": 2.740332539307413,\n      "validation": {\n        "cer": 0.19025956649719025,\n        "wer": 0.3299874529485571,\n        "exact_line_accuracy": 0.046875,\n        "samples": 256,\n        "loss": 1.808678813278675\n      }\n    },\n    {\n      "epoch": 2,\n      "train_loss": 1.3523931435157914,\n      "validation": {\n        "cer": 0.22986352689322986,\n        "wer": 0.40526976160602257,\n        "exact_line_accuracy": 0.02734375,\n        "samples": 256,\n        "loss": 1.383712537586689\n      }\n    },\n    {\n      "epoch": 3,\n      "train_loss": 1.0371341755161385,\n      "validation": {\n        "cer": 0.28418517527428416,\n        "wer": 0.4847344207444584,\n        "exact_line_accuracy": 0.00390625,\n        "samples": 256,\n        "loss": 1.2151146829128265\n      }\n    }\n  ],\n  "elapsed_seconds": 692.7065484523773,\n  "real_page_metrics": null,\n  "limitations": [\n    "Line metrics only, not real-page accuracy.",\n    "Base checkpoint already trained on IAM.",\n    "Heuristic page segmentation supports clean single-column pages."\n  ]\n}')

In [ ]:
import subprocess,sys,time,urllib.request
ui=subprocess.Popen([sys.executable,str(ROOT/"app.py")],stdout=open("/content/ui.log","w"),stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:7860",timeout=1);break
    except Exception:
        if ui.poll() is not None: raise RuntimeError(Path("/content/ui.log").read_text())
        time.sleep(1)
from google.colab import output
output.serve_kernel_port_as_iframe(7860,height=1100)